In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
cuadro = [
    cv2.cvtColor(cv2.imread(f'img/cuadro_{i}.jpg'), cv2.COLOR_BGR2RGB)
    for i in range(3)
]

udesa = [
    cv2.cvtColor(cv2.imread(f'img/udesa_{i}.jpg'), cv2.COLOR_BGR2RGB)
    for i in range(3)
]

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

for i, img in enumerate(cuadro):
    axes[0, i].imshow(img)
    axes[0, i].set_title(f'cuadro_{i}')
    axes[0, i].axis('off')

for i, img in enumerate(udesa):
    axes[1, i].imshow(img)
    axes[1, i].set_title(f'udesa_{i}')
    axes[1, i].axis('off')

plt.tight_layout()
plt.show()

## 3.1 Detección y descripción de características visuales (SIFT)

In [ ]:
sift = cv2.SIFT_create()

def detect_features(images):
    """Detecta keypoints y descriptores SIFT para una lista de imágenes."""
    kps_list, descs_list = [], []
    for img in images:
        gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
        kps, descs = sift.detectAndCompute(gray, None)
        kps_list.append(kps)
        descs_list.append(descs)
    return kps_list, descs_list

kps_udesa, descs_udesa = detect_features(udesa)
kps_cuadro, descs_cuadro = detect_features(cuadro)

for i in range(3):
    print(f"udesa_{i}: {len(kps_udesa[i])} keypoints")
for i in range(3):
    print(f"cuadro_{i}: {len(kps_cuadro[i])} keypoints")

In [ ]:
def draw_kps(img, kps, min_size=8, color=(255, 220, 0), thickness=2):
    """Dibuja keypoints con tamaño mínimo fijo para mayor visibilidad."""
    out = img.copy()
    for kp in kps:
        x, y = int(kp.pt[0]), int(kp.pt[1])
        r = max(int(kp.size / 2), min_size)
        cv2.circle(out, (x, y), r, color, thickness, cv2.LINE_AA)
    return out

fig, axes = plt.subplots(2, 3, figsize=(15, 8))

for i in range(3):
    axes[0, i].imshow(draw_kps(udesa[i], kps_udesa[i]))
    axes[0, i].set_title(f'udesa_{i} — {len(kps_udesa[i])} kps')
    axes[0, i].axis('off')

for i in range(3):
    axes[1, i].imshow(draw_kps(cuadro[i], kps_cuadro[i]))
    axes[1, i].set_title(f'cuadro_{i} — {len(kps_cuadro[i])} kps')
    axes[1, i].axis('off')

plt.suptitle('3.1 — Keypoints SIFT detectados', fontsize=14)
plt.tight_layout()
plt.show()

## 3.2 Supresión de No Máxima Adaptativa (ANMS)

In [11]:
def anms(keypoints, N):
    """
    Adaptive Non-Maximal Suppression (Algoritmo 1 del enunciado).
    Retorna los N keypoints más distribuidos espacialmente,
    priorizando los de mayor respuesta con mayor radio de supresión.
    """
    if len(keypoints) <= N:
        return keypoints

    # Extraer posiciones y respuestas
    pts = np.array([kp.pt for kp in keypoints], dtype=np.float32)   # (M, 2)
    responses = np.array([kp.response for kp in keypoints])          # (M,)

    M = len(keypoints)
    radii = np.full(M, np.inf)

    for i in range(M):
        for j in range(M):
            if responses[j] > responses[i]:
                sd = (pts[j, 0] - pts[i, 0])**2 + (pts[j, 1] - pts[i, 1])**2
                if sd < radii[i]:
                    radii[i] = sd

    # Ordenar por radio descendente y quedarse con los N mejores
    top_idx = np.argsort(radii)[::-1][:N]
    return [keypoints[i] for i in top_idx]


N = 500  # máximo de keypoints por imagen

kps_udesa_anms  = [anms(kps, N) for kps in kps_udesa]
kps_cuadro_anms = [anms(kps, N) for kps in kps_cuadro]

# Filtrar descriptores para que coincidan con los kps seleccionados
def filter_descs(kps_orig, kps_filtered, descs):
    """Devuelve los descriptores correspondientes a kps_filtered."""
    pts_orig     = [kp.pt for kp in kps_orig]
    pts_filtered = {kp.pt for kp in kps_filtered}
    idx = [i for i, pt in enumerate(pts_orig) if pt in pts_filtered]
    return descs[idx]

descs_udesa_anms  = [filter_descs(kps_udesa[i],  kps_udesa_anms[i],  descs_udesa[i])  for i in range(3)]
descs_cuadro_anms = [filter_descs(kps_cuadro[i], kps_cuadro_anms[i], descs_cuadro[i]) for i in range(3)]

for i in range(3):
    print(f"udesa_{i}:  {len(kps_udesa[i])} → {len(kps_udesa_anms[i])} kps")
for i in range(3):
    print(f"cuadro_{i}: {len(kps_cuadro[i])} → {len(kps_cuadro_anms[i])} kps")

Unexpected exception formatting exception. Falling back to standard exception


Traceback (most recent call last):
  File "C:\Users\Lucas\AppData\Roaming\Python\Python311\site-packages\IPython\core\interactiveshell.py", line 3460, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "C:\Users\Lucas\AppData\Local\Temp\ipykernel_13004\3625329562.py", line 32, in <module>
    kps_cuadro_anms = [anms(kps, N) for kps in kps_cuadro]
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Lucas\AppData\Local\Temp\ipykernel_13004\3625329562.py", line 32, in <listcomp>
    kps_cuadro_anms = [anms(kps, N) for kps in kps_cuadro]
                       ^^^^^^^^^^^^
  File "C:\Users\Lucas\AppData\Local\Temp\ipykernel_13004\3625329562.py", line -1, in anms
KeyboardInterrupt

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "C:\Users\Lucas\AppData\Roaming\Python\Python311\site-packages\IPython\core\interactiveshell.py", line 2057, in showtraceback
    stb = self.InteractiveTB.st

In [ ]:
# Comparación antes/después de ANMS
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

axes[0, 0].imshow(draw_kps(udesa[1], kps_udesa[1]))
axes[0, 0].set_title(f'udesa_1 — SIFT ({len(kps_udesa[1])} kps)')
axes[0, 0].axis('off')

axes[0, 1].imshow(draw_kps(udesa[1], kps_udesa_anms[1]))
axes[0, 1].set_title(f'udesa_1 — ANMS ({len(kps_udesa_anms[1])} kps)')
axes[0, 1].axis('off')

axes[1, 0].imshow(draw_kps(cuadro[1], kps_cuadro[1]))
axes[1, 0].set_title(f'cuadro_1 — SIFT ({len(kps_cuadro[1])} kps)')
axes[1, 0].axis('off')

axes[1, 1].imshow(draw_kps(cuadro[1], kps_cuadro_anms[1]))
axes[1, 1].set_title(f'cuadro_1 — ANMS ({len(kps_cuadro_anms[1])} kps)')
axes[1, 1].axis('off')

plt.suptitle('3.2 — SIFT vs ANMS (imagen ancla)', fontsize=14)
plt.tight_layout()
plt.show()

## 3.3 Asociación de características (Matching)

In [ ]:
def lowe_match(desc1, desc2, ratio=0.65):
    """
    Matching con Lowe's ratio test + cross-check.
    - knnMatch k=2 con ratio test (ratio < threshold)
    - Cross-check: solo mantiene matches que son mutuamente los mejores
    """
    bf = cv2.BFMatcher(cv2.NORM_L2)

    # Lowe's ratio en ambas direcciones
    knn_12 = bf.knnMatch(desc1, desc2, k=2)
    knn_21 = bf.knnMatch(desc2, desc1, k=2)

    good_12 = {m.queryIdx: m for m, n in knn_12 if m.distance < ratio * n.distance}
    good_21 = {m.queryIdx: m for m, n in knn_21 if m.distance < ratio * n.distance}

    # Cross-check: m(i→j) válido solo si también j→i apunta a i
    matches = []
    for i, m in good_12.items():
        j = m.trainIdx
        if j in good_21 and good_21[j].trainIdx == i:
            matches.append(m)

    return matches

# Ancla = imagen _1 en ambos conjuntos
matches_udesa_01  = lowe_match(descs_udesa_anms[0],  descs_udesa_anms[1])
matches_udesa_21  = lowe_match(descs_udesa_anms[2],  descs_udesa_anms[1])
matches_cuadro_01 = lowe_match(descs_cuadro_anms[0], descs_cuadro_anms[1])
matches_cuadro_21 = lowe_match(descs_cuadro_anms[2], descs_cuadro_anms[1])

print(f"udesa  0→1: {len(matches_udesa_01)}  matches")
print(f"udesa  2→1: {len(matches_udesa_21)}  matches")
print(f"cuadro 0→1: {len(matches_cuadro_01)} matches")
print(f"cuadro 2→1: {len(matches_cuadro_21)} matches")

In [ ]:
# Visualización de matches con líneas más visibles
def show_matches(img1, kps1, img2, kps2, matches, title, max_draw=60):
    # Construir imagen side-by-side manualmente para controlar colores y grosor
    h1, w1 = img1.shape[:2]
    h2, w2 = img2.shape[:2]
    h = max(h1, h2)
    canvas = np.zeros((h, w1 + w2, 3), dtype=np.uint8)
    canvas[:h1, :w1]      = img1
    canvas[:h2, w1:w1+w2] = img2

    subset = matches[:max_draw]
    np.random.seed(42)
    colors = [tuple(int(c) for c in np.random.randint(50, 255, 3)) for _ in subset]

    for match, color in zip(subset, colors):
        pt1 = tuple(map(int, kps1[match.queryIdx].pt))
        pt2 = (int(kps2[match.trainIdx].pt[0]) + w1,
               int(kps2[match.trainIdx].pt[1]))
        cv2.line(canvas, pt1, pt2, color, thickness=1, lineType=cv2.LINE_AA)
        cv2.circle(canvas, pt1, 4, color, -1, cv2.LINE_AA)
        cv2.circle(canvas, pt2, 4, color, -1, cv2.LINE_AA)

    plt.figure(figsize=(16, 5))
    plt.imshow(canvas)
    plt.title(f'{title}  —  {len(matches)} matches  (mostrando {min(max_draw, len(matches))})',
              fontsize=12)
    plt.axis('off')
    plt.tight_layout()
    plt.show()

show_matches(udesa[0],  kps_udesa_anms[0],  udesa[1],  kps_udesa_anms[1],  matches_udesa_01,  'udesa 0→1')
show_matches(udesa[2],  kps_udesa_anms[2],  udesa[1],  kps_udesa_anms[1],  matches_udesa_21,  'udesa 2→1')
show_matches(cuadro[0], kps_cuadro_anms[0], cuadro[1], kps_cuadro_anms[1], matches_cuadro_01, 'cuadro 0→1')
show_matches(cuadro[2], kps_cuadro_anms[2], cuadro[1], kps_cuadro_anms[1], matches_cuadro_21, 'cuadro 2→1')

## 3.4 Estimación de la homografía "manualmente" (DLT)

In [ ]:
def pick_points(img1, img2, n=4, label1='Imagen 1', label2='Imagen 2'):
    """
    Muestra img1 e img2 side-by-side y permite clickear n puntos en cada una.
    Retorna pts1 (n,2) y pts2 (n,2) en coordenadas de cada imagen original.
    Instrucciones: clickear los n puntos en img1 (izquierda), luego los n en img2 (derecha).
    """
    h1, w1 = img1.shape[:2]
    h2, w2 = img2.shape[:2]
    h = max(h1, h2)
    canvas = np.zeros((h, w1 + w2, 3), dtype=np.uint8)
    canvas[:h1, :w1]      = img1
    canvas[:h2, w1:w1+w2] = img2

    fig, ax = plt.subplots(figsize=(16, 6))
    ax.imshow(canvas)
    ax.axvline(w1, color='yellow', linewidth=2)
    ax.set_title(f'Clickeá {n} puntos en "{label1}" (izq) y luego {n} en "{label2}" (der)\n'
                 f'Total de clicks: {2*n}', fontsize=11)
    plt.tight_layout()

    clicks = plt.ginput(2 * n, timeout=120)
    plt.close()

    pts1 = np.array([[x,     y] for x, y in clicks[:n]],           dtype=np.float64)
    pts2 = np.array([[x - w1, y] for x, y in clicks[n:2*n]],       dtype=np.float64)
    return pts1, pts2


# ── Selector interactivo — ejecutar UNA VEZ por par y guardar los puntos ──────
# Descomentar el par que querés calibrar, ejecutar la celda, clickear los puntos.
# Luego los resultados quedan en pts_src / pts_dst para usar en la celda siguiente.

# --- udesa 0 → 1 ---
# pts_udesa_0_src, pts_udesa_0_dst = pick_points(udesa[0], udesa[1], label1='udesa_0', label2='udesa_1')

# --- udesa 2 → 1 ---
# pts_udesa_2_src, pts_udesa_2_dst = pick_points(udesa[2], udesa[1], label1='udesa_2', label2='udesa_1')

# --- cuadro 0 → 1 ---
# pts_cuadro_0_src, pts_cuadro_0_dst = pick_points(cuadro[0], cuadro[1], label1='cuadro_0', label2='cuadro_1')

# --- cuadro 2 → 1 ---
# pts_cuadro_2_src, pts_cuadro_2_dst = pick_points(cuadro[2], cuadro[1], label1='cuadro_2', label2='cuadro_1')

print("Descomentá el par que querés calibrar, ejecutá la celda y clickeá los 8 puntos (4+4).")
print("Una vez calibrados todos los pares, pasá a la celda siguiente.")

In [ ]:
# Verificación visual DLT: overlay semi-transparente warped sobre ancla
def show_warp(img_src, img_anchor, H, title):
    h, w = img_anchor.shape[:2]
    warped = cv2.warpPerspective(img_src, H, (w, h))

    # Overlay: warped en rojo, ancla en cian — donde coinciden se ve blanco/gris
    overlay = np.zeros((h, w, 3), dtype=np.uint8)
    overlay[..., 0] = warped[..., 0]                        # R ← warped
    overlay[..., 1] = img_anchor[..., 1] // 2              # G ← ancla (dim)
    overlay[..., 2] = img_anchor[..., 2] // 2              # B ← ancla (dim)

    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    axes[0].imshow(img_src);    axes[0].set_title('Fuente');          axes[0].axis('off')
    axes[1].imshow(img_anchor); axes[1].set_title('Ancla');           axes[1].axis('off')
    axes[2].imshow(warped);     axes[2].set_title('Warped (DLT)');    axes[2].axis('off')
    plt.suptitle(title, fontsize=13)
    plt.tight_layout()
    plt.show()

    # Overlay para verificar alineamiento
    fig2, ax = plt.subplots(figsize=(10, 4))
    ax.imshow(cv2.addWeighted(img_anchor, 0.5, warped, 0.5, 0))
    ax.set_title(f'{title} — overlay 50/50 (ancla + warped)')
    ax.axis('off')
    plt.tight_layout()
    plt.show()

show_warp(udesa[0],  udesa[1],  H_dlt_udesa_01,  '3.4 DLT — udesa 0→1')
show_warp(udesa[2],  udesa[1],  H_dlt_udesa_21,  '3.4 DLT — udesa 2→1')
show_warp(cuadro[0], cuadro[1], H_dlt_cuadro_01, '3.4 DLT — cuadro 0→1')
show_warp(cuadro[2], cuadro[1], H_dlt_cuadro_21, '3.4 DLT — cuadro 2→1')